In [1]:
import pandas as pd

In [2]:
dfs_einstein = pd.read_excel("../data/EINSTEIN/20230724_Dados_Epidemiologicos_Semanais_ITpS_SE29.xlsx", sheet_name=None)
dfs_einstein = [df.assign(sheetName=sheet) for sheet, df in dfs_einstein.items() if not sheet.startswith("EXAMES")]
df_einstein = pd.concat(dfs_einstein, ignore_index=True)

## Fix Data Einstein

In [3]:
import hashlib

In [4]:
PATHOGEN_COLUMNS = [
    "COVID",
    "DENGUE",
    "INFLUENZA A",
    "INFLUENZA B",
    "VARIOLA SIMIA",
    "VIRUS SINCICIAL RESPIRATÓRIO",
]

ID_COLUMNS = [
    # id columns
    "ACCESSION",
    "EXAME",
    "DETALHE_EXAME",
    "SEXO",
    "IDADE",
    "MUNICÍPIO",
    "ESTADO",
    "DH_COLETA",
]

In [5]:
df_einstein.shape

(714617, 11)

In [6]:
df_einstein = df_einstein.query("PATOGENO not in ('DENGUE', 'VARIOLA SIMIA')")

In [7]:
df_einstein_pivot = (
    df_einstein.assign(
        DETALHE_EXAME=lambda df: df["DETALHE_EXAME"].fillna("S/ DETALHE"),
    )
    .assign(
        RESULTADO=lambda df: df["RESULTADO"]
        .str.lower()
        .str.strip()
        .map(
            {
                "não detectado": 0,#"Neg",
                "detectado": 1,#"Pos",
            }
        )
    )
    .assign(
        # CORRIGINDO PATOGENO INFLUENZA
        PATOGENO=lambda df: df.apply(
            lambda row: "INFLUENZA B"
            if "INFLUENZA B" in row["DETALHE_EXAME"]
            else "INFLUENZA A"
            if "INF A" in row["DETALHE_EXAME"] or "INFLUENZA" in row["DETALHE_EXAME"]
            else row["PATOGENO"],
            axis=1,
        )
    )
    .pivot_table(
        index=ID_COLUMNS,
        columns="PATOGENO",
        values="RESULTADO",
        aggfunc="first",
        fill_value=-1#"NT"
    )
    .reset_index()
)

In [12]:
(
    df_einstein.assign(
        DETALHE_EXAME=lambda df: df["DETALHE_EXAME"].fillna("S/ DETALHE"),
    )
    .assign(
        RESULTADO=lambda df: df["RESULTADO"]
        .str.lower()
        .str.strip()
        .map(
            {
                "não detectado": 0,#"Neg",
                "detectado": 1,#"Pos",
            }
        )
    )
    .assign(
        # CORRIGINDO PATOGENO INFLUENZA
        PATOGENO=lambda df: df.apply(
            lambda row: "INFLUENZA B"
            if "INFLUENZA B" in row["DETALHE_EXAME"]
            else "INFLUENZA A"
            if "INF A" in row["DETALHE_EXAME"] or "INFLUENZA" in row["DETALHE_EXAME"]
            else row["PATOGENO"],
            axis=1,
        )
    )
    .assign(
        **{
            pathogen: lambda df: df['RESULTADO'].mask(df['PATOGENO'] == pathogen, 'NT')
            for pathogen in ['COVID', 'INFLUENZA A', 'INFLUENZA B', 'VIRUS SINCICIAL RESPIRATÓRIO']
        }
    )
)

,ACCESSION,SEXO,IDADE,EXAME,DH_COLETA,MUNICÍPIO,ESTADO,PATOGENO,RESULTADO,sheetName,DETALHE_EXAME,COVID,INFLUENZA A,INFLUENZA B,VIRUS SINCICIAL RESPIRATÓRIO
0,2023204005355,FEMININO,71,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,COVID,0,COVID-19,S/ DETALHE,0,0,0,0
1,2023205006395,MASCULINO,46,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,COVID,0,COVID-19,S/ DETALHE,0,0,0,0
2,2023205006494,MASCULINO,61,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,COVID,0,COVID-19,S/ DETALHE,0,0,0,0
3,2023205007748,MASCULINO,0,PCR PAINEL DE PATOGENOS RESPIRATORIO,2023-07-24,SÃO PAULO,SÃO PAULO,COVID,0,COVID-19,S/ DETALHE,0,0,0,0
4,2023205008704,FEMININO,80,TESTE RÁPIDO-ANTÍGENO COVID-19 (SARS,2023-07-24,SÃO PAULO,SÃO PAULO,COVID,0,COVID-19,S/ DETALHE,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714612,2022001000793,FEMININO,1,PCR PARA INFLUENZA A/B E VRS,2022-01-01,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,0,VSR,VÍRUS SINCICIAL RESPIRATÓRIO,NT,NT,NT,NT
714613,2022001001981,FEMININO,1,PCR PARA INFLUENZA A/B E VRS,2022-01-01,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,1,VSR,VÍRUS SINCICIAL RESPIRATÓRIO,NT,NT,NT,NT
714614,2022001003087,FEMININO,75,PCR PARA INFLUENZA A/B E VRS,2022-01-01,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,0,VSR,VÍRUS SINCICIAL RESPIRATÓRIO,NT,NT,NT,NT
714615,2022001003240,MASCULINO,64,PCR PARA INFLUENZA A/B E VRS,2022-01-01,SÃO PAULO,SÃO PAULO,VIRUS SINCICIAL RESPIRATÓRIO,1,VSR,VÍRUS SINCICIAL RESPIRATÓRIO,NT,NT,NT,NT


In [8]:
df_einstein_pivot.shape

(679849, 12)

In [10]:
ID_AGG_COLUMNS = ['ACCESSION', 'EXAME']
TEST_RESULT_COLUMNS = ['COVID', 'INFLUENZA A', 'INFLUENZA B', 'VIRUS SINCICIAL RESPIRATÓRIO']
NON_ID_AGG_MAX_COLUMNS = ['DH_COLETA', 'IDADE', 'SEXO', 'MUNICÍPIO', 'ESTADO'] 


df_einstein_agg = (
    df_einstein_pivot
    .groupby(ID_AGG_COLUMNS)
    .agg(
        {
            **{col: 'max' for col in NON_ID_AGG_MAX_COLUMNS},
            **{
                test_result_column: 'max' 
                for test_result_column in TEST_RESULT_COLUMNS
            }
        }
    )
    .reset_index()
)

In [ ]:
[
    "lab_id",
    "test_id",
    "test_kit",
    "patient_id",
    "sample_id",
    "state",
    "location",
    "date_testing",
    "epiweek",
    "age",
    "sex",
    "FLUA_test_result",
    "Ct_FluA",
    "FLUB_test_result",
    "Ct_FluB",
    "VSR_test_result",
    "Ct_VSR",
    "SC2_test_result",
    "Ct_geneE",
    "Ct_geneN",
    "Ct_geneS",
    "Ct_ORF1ab",
    "Ct_RDRP",
    "geneS_detection",
    "META_test_result",
    "RINO_test_result",
    "PARA_test_result",
    "ADENO_test_result",
    "BOCA_test_result",
    "COVS_test_result",
    "ENTERO_test_result",
    "BAC_test_result",
]

['lab_id',
 'test_id',
 'test_kit',
 'patient_id',
 'sample_id',
 'state',
 'location',
 'date_testing',
 'epiweek',
 'age',
 'sex',
 'FLUA_test_result',
 'Ct_FluA',
 'FLUB_test_result',
 'Ct_FluB',
 'VSR_test_result',
 'Ct_VSR',
 'SC2_test_result',
 'Ct_geneE',
 'Ct_geneN',
 'Ct_geneS',
 'Ct_ORF1ab',
 'Ct_RDRP',
 'geneS_detection',
 'META_test_result',
 'RINO_test_result',
 'PARA_test_result',
 'ADENO_test_result',
 'BOCA_test_result',
 'COVS_test_result',
 'ENTERO_test_result',
 'BAC_test_result']

In [ ]:
df_einstein_agg.rename(
    columns = {
        "INFLUENZA A": "FLUA_test_result",
        "INFLUENZA B": "FLUB_test_result",
        "VIRUS SINCICIAL RESPIRATÓRIO": "VSR_test_result",
        "COVID": "SC2_test_result",

        "ACCESSION": "test_id",
        "EXAME": "test_kit",

        "DH_COLETA": "date_testing",
        "ESTADO":"state",
        "MUNICIPIO":"location",
        "IDADE": "age",
    }
)

PATOGENO,test_id,test_kit,date_testing,age,SEXO,MUNICÍPIO,state,SC2_test_result,DENGUE,FLUA_test_result,FLUB_test_result,VARIOLA SIMIA,VSR_test_result
0,2021365008664,OPERAÇÃO AEROPORTO PCR COVID-19,2022-01-01,61,MASCULINO,GUARULHOS,SÃO PAULO,Neg,NT,NT,NT,NT,NT
1,2022001000003,PCR EM TEMPO REAL PARA DETECÇÃO DE C,2022-01-01,53,FEMININO,SÃO PAULO,SÃO PAULO,Neg,NT,NT,NT,NT,NT
2,2022001000004,PCR EM TEMPO REAL PARA DETECÇÃO DE C,2022-01-01,61,MASCULINO,SÃO PAULO,SÃO PAULO,Neg,NT,NT,NT,NT,NT
3,2022001000005,PCR EM TEMPO REAL PARA DETECÇÃO DE C,2022-01-01,58,MASCULINO,SÃO PAULO,SÃO PAULO,Neg,NT,NT,NT,NT,NT
4,2022001000006,PCR EM TEMPO REAL PARA DETECÇÃO DE C,2022-01-01,49,FEMININO,SÃO PAULO,SÃO PAULO,Neg,NT,NT,NT,NT,NT
...,...,...,...,...,...,...,...,...,...,...,...,...,...
476905,2023205008692,PESQUISA RÁPIDA PARA INFLUENZA A E B,2023-07-24,14,FEMININO,SÃO PAULO,SÃO PAULO,NT,NT,Neg,Neg,NT,NT
476906,2023205008704,TESTE RÁPIDO-ANTÍGENO COVID-19 (SARS,2023-07-24,80,FEMININO,SÃO PAULO,SÃO PAULO,Neg,NT,NT,NT,NT,NT
476907,2023205008906,PESQUISA RÁPIDA PARA INFLUENZA A E B,2023-07-24,28,FEMININO,SÃO PAULO,SÃO PAULO,NT,NT,Neg,Neg,NT,NT
476908,2023205008927,PESQUISA RÁPIDA PARA INFLUENZA A E B,2023-07-24,1,MASCULINO,SÃO PAULO,SÃO PAULO,NT,NT,Neg,Neg,NT,NT
